In [27]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [28]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000013211B02FB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013211B02B00>, model_name='llama-3.1-8b-instant')

In [29]:
from langchain_core.messages import HumanMessage
model.invoke([
    HumanMessage(content="Hi, My name is Sejal and I am learning AI Engineering")
])

AIMessage(content='Nice to meet you, Sejal. AI Engineering is a fascinating field that combines computer science, engineering, and machine learning principles to design, develop, and deploy intelligent systems. What specific aspects of AI Engineering are you interested in learning more about? Are you focusing on areas like Natural Language Processing, Computer Vision, Reinforcement Learning, or something else?', response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 48, 'total_tokens': 119, 'completion_time': 0.107338251, 'completion_tokens_details': None, 'prompt_time': 0.00347837, 'prompt_tokens_details': None, 'queue_time': 0.051474934, 'total_time': 0.110816621}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-1ec72f25-0afa-4e5d-97b5-63dffcdb548f-0', usage_metadata={'input_tokens': 48, 'output_tokens': 71, 'total_tokens': 119})

In [30]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hi, My name is Sejal and I am learning AI Engineering"),
    AIMessage(content="Hello Sejal, nice to meet you. AI Engineering is a fascinating field that combines the principles of software engineering with the power of artificial intelligence. It's exciting that you're learning about it.\n\nWhat specifically are you looking to learn or achieve in AI Engineering? Are you focusing on a particular area such as natural language processing, computer vision, or reinforcement learning? Or are you looking to learn the fundamentals of AI and machine learning?\n\nI'm here to help and provide any guidance or resources you might need. What's on your mind?"),
    HumanMessage(content="Hey, what is my name and what I do?")
])

AIMessage(content='Your name is Sejal and you are learning AI Engineering.', response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 176, 'total_tokens': 189, 'completion_time': 0.02575572, 'completion_tokens_details': None, 'prompt_time': 0.012973978, 'prompt_tokens_details': None, 'queue_time': 0.051592996, 'total_time': 0.038729698}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-58cd4052-391f-46da-b6a9-b7a9c36cceeb-0', usage_metadata={'input_tokens': 176, 'output_tokens': 13, 'total_tokens': 189})

In [31]:
### Message History
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [32]:
config={"configurable":{"session_id":"chat1"}}

In [33]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Sejal and I am learning AI Engineering")],
    config=config
)

In [34]:
response.content

"Nice to meet you, Sejal. AI Engineering is an exciting field with a wide range of applications and opportunities. What specific areas of AI Engineering are you interested in or currently learning about? Are you looking to build conversational interfaces, computer vision systems, or something else?\n\nAs you learn and grow in this field, I'd be happy to provide guidance, answer questions, or discuss topics related to AI Engineering. What's your current level of experience with AI? Are you a beginner, intermediate, or advanced learner?"

In [35]:
with_message_history.invoke(
    [HumanMessage(content="Whats my name?")],
    config=config
)

AIMessage(content='Your name is Sejal.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 166, 'total_tokens': 173, 'completion_time': 0.005526248, 'completion_tokens_details': None, 'prompt_time': 0.012753052, 'prompt_tokens_details': None, 'queue_time': 0.078249819, 'total_time': 0.0182793}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-240846ea-66a4-4641-b55d-4e97d20d0359-0', usage_metadata={'input_tokens': 166, 'output_tokens': 7, 'total_tokens': 173})

In [36]:
### change the config--->i.e change session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name?")],
    config=config1
)
response.content

"I don't have any information about your name. This is the start of our conversation, and I don't retain any information about individual users. If you'd like to share your name, I'd be happy to chat with you."

In [37]:
response=with_message_history.invoke(
    [HumanMessage(content="My name is Sejal")],
    config=config1
)
response.content

'Nice to meet you, Sejal. How are you today? Is there something I can help you with or would you like to chat about your interests?'

In [38]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name?")],
    config=config1
)
response.content

'Your name is Sejal.'

In [39]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are helpfull assistant.Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt |model

In [40]:
chain.invoke({"messages": [HumanMessage(content="Hi My name is Sejal")]})

AIMessage(content="Nice to meet you, Sejal. I'm happy to be your assistant. How can I assist you today?", response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 58, 'total_tokens': 82, 'completion_time': 0.596473762, 'completion_tokens_details': None, 'prompt_time': 0.00395997, 'prompt_tokens_details': None, 'queue_time': 0.049668355, 'total_time': 0.600433732}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-aef1fd87-871a-4c2a-8ab3-7f81111211c9-0', usage_metadata={'input_tokens': 58, 'output_tokens': 24, 'total_tokens': 82})

In [41]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [42]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Sejal")],
    config=config
)
response.content

"Nice to meet you, Sejal! I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far?"

In [43]:
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are helpfull assistant.Answer all the questions to the best of your ability in {languages}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt |model

In [44]:
response=chain.invoke(
    {"messages": [HumanMessage(content="Hi My name is Sejal")], 
     "languages": ["Marathi"]}
)
response.content

'नमस्कार, तुमचे नाव सेजल आहे का?'

In [45]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [46]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi I am Sejal")], "languages": ["Marathi"]},
    config=config
)
response.content

'नमस्कार! आणि तुमचा वेलकम आहे! मी तुमच्या मदतील आहे, काही विचारले का?'

In [47]:
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")], "languages": ["Marathi"]},
    config=config
)
response.content

'तुमचे नाव "सेजल" आहे.'

Managing Conversation History
1.trim_messages-->helper to reduce how many messages we are sending to the model.

In [51]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=40,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="You are a good assistant."),
    HumanMessage(content="Hi, I am Sejal"),
    AIMessage(content="Hi"),
    HumanMessage(content="I like chocolate ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="What is 2+2?"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks"),
    AIMessage(content="No problem"),
    HumanMessage(content="Having fun"),
    AIMessage(content="yes!"),   
]
trimmer.invoke(messages)

c:\Users\Sejal Pc\Desktop\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sejal Pc\Desktop\Langchain\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sejal Pc\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer

[SystemMessage(content='You are a good assistant.'),
 HumanMessage(content='What is 2+2?'),
 AIMessage(content='4'),
 HumanMessage(content='Thanks'),
 AIMessage(content='No problem'),
 HumanMessage(content='Having fun'),
 AIMessage(content='yes!')]

In [56]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt| model
)

response=chain.invoke(
    {"messages":messages+[HumanMessage(content="What icecream do i like?")],
    "languages":"Marathi"
    }
)
response.content

'मी जाणून घेईल! तुम्ही कुठल्या प्रकारचे आइस क्रीम आवडते? व्हॅनिला, चॉकलेट किंवा काही इतर?'

In [57]:
response=chain.invoke(
    {"messages":messages+[HumanMessage(content="What personal question did i ask you?")],
    "languages":"Marathi"
    }
)
response.content

'तुला काही वैयक्तिक प्रश्न विचारले नाही होते.'